# Data Engineering Interview Prep: Easy Python (Complete 1-15)
## Topic: Arrays, Hash Sets, Strings, and Time/Space Complexity

---

## Table of Contents
1. [Google: Fizz Buzz Sum](#python-lesson-1-google---fizz-buzz-sum)
2. [Amazon: Intersection of Two Lists](#python-lesson-2-amazon---intersection-of-two-lists)
3. [Apple: Contains Duplicate](#python-lesson-3-apple---contains-duplicate)
4. [Workday: Is Anagram?](#python-lesson-4-workday---is-anagram)
5. [Spotify: Is Palindrome](#python-lesson-5-spotify---is-palindrome)
6. [Palantir: Roman to Integer](#python-lesson-6-palantir---roman-to-integer)
7. [Uber: Pascal's Triangle](#python-lesson-7-uber---pascals-triangle)
8. [ServiceNow: Same Stripes](#python-lesson-8-servicenow---same-stripes-toeplitz-matrix)
9. [Capital One: Base 13 Conversion](#python-lesson-9-capital-one---base-13-conversion)
10. [Microsoft: Factorial Formula](#python-lesson-10-microsoft---factorial-formula)
11. [Spotify: Another One](#python-lesson-11-spotify---another-one-plus-one)
12. [Intuit: Weakest Strong Link](#python-lesson-12-intuit---weakest-strong-link)
13. [Fintech: Compound Interest](#python-lesson-13-fintech---compound-interest)
14. [TikTok: Triangular Sum](#python-lesson-14-tiktok---triangular-sum)
15. [Tesla: Counting Letters In Numbers](#python-lesson-15-tesla---counting-letters-in-numbers)

---

### **Python Lesson 1: Google - "Fizz Buzz Sum"**

#### **The Problem**
Write a function `fizz_buzz_sum(target)` that returns the sum of all positive integers strictly less than `target` that are divisible by either 3 or 5.

#### **The Logic ($O(N)$ Time, $O(1)$ Space)**
Use a Pythonic generator expression inside the built-in `sum()` function.

#### **The Solution (Python 3)**
```python
def fizz_buzz_sum(target: int) -> int:
    return sum(i for i in range(1, target) if i % 3 == 0 or i % 5 == 0)
```

#### **Senior Data Engineer Perspective**
* **The PySpark UDF Trap:** Wrapping this in a PySpark `udf()` forces Spark to serialize data between the JVM and Python processes, destroying performance. Use native Spark SQL column math instead: `df.withColumn("is_fizzbuzz", F.when((F.col("num") % 3 == 0) | (F.col("num") % 5 == 0), 1).otherwise(0))`.

---

### **Python Lesson 2: Amazon - "Intersection of Two Lists"**

#### **The Problem**
Write a function `intersection(a, b)` that returns a list of the unique elements that exist in both lists.

#### **The Logic ($O(N + M)$ Time, $O(N + M)$ Space)**
By converting the lists to Hash Sets, lookups become $O(1)$ constant time, and we can use the bitwise `&` operator.

#### **The Solution (Python 3)**
```python
def intersection(a: list[int], b: list[int]) -> list[int]:
    return list(set(a) & set(b))
```

#### **Senior Data Engineer Perspective**
* **Memory Limits (OOM):** `set(a)` pulls the entire list into the memory of a single machine. For massive datasets, use Databricks/Spark DataFrames and execute an `inner join` or native `.intersect()` to orchestrate a network shuffle without blowing up a single node's RAM.

---

### **Python Lesson 3: Apple - "Contains Duplicate"**

#### **The Problem**
Write a function `contains_duplicate(nums)` that returns `True` if any value appears at least twice in the array.

#### **The Logic ($O(N)$ Time, $O(N)$ Space)**
Convert the list to a set (which strips duplicates) and compare the length to the original list.

#### **The Solution (Python 3)**
```python
def contains_duplicate(nums: list[int]) -> bool:
    return len(nums) != len(set(nums))
```

#### **Senior Data Engineer Perspective**
* **Approximate Distinct Counting:** Checking exact duplicates across petabytes of data requires a massive network shuffle. A Senior DE uses **HyperLogLog** (`approx_count_distinct` in Spark) to estimate unique visitors with 95%+ accuracy using only kilobytes of memory.

---

### **Python Lesson 4: Workday - "Is Anagram?"**

#### **The Problem**
Write a function `is_anagram(s, t)` that returns `True` if `t` is an anagram of `s`.

#### **The Logic ($O(N)$ Time, $O(N)$ Space)**
Avoid sorting ($O(N \log N)$). Achieve $O(N)$ time by counting character frequency using Python's highly optimized `Counter` class.

#### **The Solution (Python 3)**
```python
from collections import Counter

def is_anagram(s: str, t: str) -> bool:
    if len(s) != len(t):
        return False
    return Counter(s) == Counter(t)
```

#### **Senior Data Engineer Perspective**
* **Failing Fast:** The `if len(s) != len(t):` guard clause is critical. Writing cheap boolean checks that safely drop invalid data before applying heavy computational logic saves significant compute costs at scale.

---

### **Python Lesson 5: Spotify - "Is Palindrome"**

#### **The Problem**
Determine if a string `s` reads the same forwards and backwards, ignoring spaces, punctuation, and capitalization.

#### **The Logic ($O(N)$ Time, $O(1)$ Space)**
Use **Two Pointers** (start and end) that move inward, comparing characters in place, to avoid copying the string in memory.

#### **The Solution (Python 3)**
```python
def is_palindrome(s: str) -> bool:
    left, right = 0, len(s) - 1
    
    while left < right:
        while left < right and not s[left].isalnum():
            left += 1
        while left < right and not s[right].isalnum():
            right -= 1
            
        if s[left].lower() != s[right].lower():
            return False
            
        left += 1
        right -= 1
        
    return True
```

#### **Senior Data Engineer Perspective**
* **In-Place Operations:** Duplicating 500,000 text documents in memory just to reverse them will trigger severe Garbage Collection (GC) pauses in Spark. Two-pointer strategies avoid this memory bloat.

---

### **Python Lesson 6: Palantir - "Roman to Integer"**

#### **The Problem**
Convert a Roman numeral string to an integer.

#### **The Logic ($O(N)$ Time, $O(1)$ Space)**
Use a Hash Map for symbol values. If a symbol is smaller than the next symbol, subtract it; otherwise, add it.

#### **The Solution (Python 3)**
```python
def roman_to_int(s: str) -> int:
    roman_map = {'I': 1, 'V': 5, 'X': 10, 'L': 50, 'C': 100, 'D': 500, 'M': 1000}
    total = 0
    
    for i in range(len(s)):
        if i + 1 < len(s) and roman_map[s[i]] < roman_map[s[i+1]]:
            total -= roman_map[s[i]]
        else:
            total += roman_map[s[i]]
            
    return total
```

#### **Senior Data Engineer Perspective**
* **Broadcast Variables:** If building a PySpark UDF, do not instantiate `roman_map` inside the function. Use `sc.broadcast(roman_map)` to send a read-only copy to each worker node once, preventing millions of dictionary recreations per partition.

---

### **Python Lesson 7: Uber - "Pascal's Triangle"**

#### **The Problem**
Given `numRows`, return the first `numRows` of Pascal's triangle.

#### **The Logic ($O(N^2)$ Time, $O(N^2)$ Space)**
Build row by row, making the inner elements the sum of the two elements directly above.

#### **The Solution (Python 3)**
```python
def generate_pascals_triangle(numRows: int) -> list[list[int]]:
    if numRows == 0: return []
    triangle = [[1]]
    
    for i in range(1, numRows):
        prev_row = triangle[-1]
        new_row = [1]
        for j in range(1, i):
            new_row.append(prev_row[j-1] + prev_row[j])
        new_row.append(1)
        triangle.append(new_row)
        
    return triangle
```

#### **Senior Data Engineer Perspective**
* **Sequential Dependencies:** Row 5 cannot be computed until Row 4 is finished. Recognizing when an algorithm forces data to be processed iteratively on a single machine (the Driver node) is a crucial skill for scaling pipelines.

---

### **Python Lesson 8: ServiceNow - "Same Stripes" (Toeplitz Matrix)**

#### **The Problem**
Given an `M x N` matrix, return `True` if every diagonal from top-left to bottom-right has the same elements.

#### **The Logic ($O(M \times N)$ Time, $O(1)$ Space)**
Check if every element `matrix[i][j]` equals `matrix[i+1][j+1]`.

#### **The Solution (Python 3)**
```python
def is_toeplitz_matrix(matrix: list[list[int]]) -> bool:
    rows, cols = len(matrix), len(matrix[0])
    
    for r in range(rows - 1):
        for c in range(cols - 1):
            if matrix[r][c] != matrix[r + 1][c + 1]:
                return False
    return True
```

#### **Senior Data Engineer Perspective**
* **Matrix Data Structures:** Standard Python lists of lists are slow. Real data pipelines use **NumPy** or **Apache Arrow**, which store matrix data in contiguous memory blocks, allowing for vectorized operations executed in C.

---

### **Python Lesson 9: Capital One - "Base 13 Conversion"**

#### **The Problem**
Convert a base-10 integer to a base-13 string.

#### **The Logic ($O(\log N)$ Time, $O(\log N)$ Space)**
Repeatedly divide by 13, taking the remainder for the digit.

#### **The Solution (Python 3)**
```python
def convert_to_base_13(num: int) -> str:
    if num == 0: return "0"
    chars = "0123456789abc"
    result = []
    is_negative = num < 0
    num = abs(num)
    
    while num > 0:
        result.append(chars[num % 13])
        num //= 13
        
    if is_negative: result.append("-")
    return "".join(result[::-1])
```

#### **Senior Data Engineer Perspective**
* **String Concatenation Penalty:** Python strings are immutable. Using `result_string += char` in a loop creates a new string in memory every iteration. Appending to a list and using `.join()` is exponentially faster.

---

### **Python Lesson 10: Microsoft - "Factorial Formula"**

#### **The Problem**
Return the factorial of `n`.

#### **The Solution (Python 3)**
```python
def factorial(n: int) -> int:
    result = 1
    for i in range(2, n + 1):
        result *= i
    return result
```

---

### **Python Lesson 11: Spotify - "Another One" (Plus One)**

#### **The Problem**
Increment a large integer represented as an array of digits by one.

#### **The Solution (Python 3)**
```python
def plus_one(digits: list[int]) -> list[int]:
    for i in range(len(digits) - 1, -1, -1):
        if digits[i] < 9:
            digits[i] += 1
            return digits
        digits[i] = 0
    return [1] + digits
```

---

### **Python Lesson 12: Intuit - "Weakest Strong Link"**

#### **The Problem**
Find the minimum element in an array.

#### **The Solution (Python 3)**
```python
def find_minimum(nums: list[int]) -> int:
    if not nums: raise ValueError("Empty list")
    min_val = nums[0]
    for num in nums[1:]:
        if num < min_val:
            min_val = num
    return min_val
```
**Senior DE Perspective:** Never use native Python loops for min/max aggregations on large datasets. PySpark's `df.select(F.min('col'))` executes this in parallel.

---

### **Python Lesson 13: Fintech - "Compound Interest"**

#### **The Problem**
Calculate compound interest: $A = P(1 + r/n)^{nt}$.

#### **The Solution (Python 3)**
```python
def compound_interest(principal: float, rate: float, time: int, n: int) -> float:
    return round(principal * ((1 + (rate / n)) ** (n * time)), 2)
```
**Senior DE Perspective:** Dealing with floats in financial equations causes precision loss. Strictly use Python's `decimal` library (or `DecimalType` in PySpark) for all monetary calculations.

---

### **Python Lesson 14: TikTok - "Triangular Sum"**

#### **The Problem**
Repeatedly replace an array with a new array where `new_nums[i] = (nums[i] + nums[i+1]) % 10`. Return the final digit.

#### **The Solution (Python 3)**
```python
def triangular_sum(nums: list[int]) -> int:
    while len(nums) > 1:
        nums = [(nums[i] + nums[i+1]) % 10 for i in range(len(nums) - 1)]
    return nums[0]
```

---

### **Python Lesson 15: Tesla - "Counting Letters In Numbers"**

#### **The Problem**
Count the total number of letters used to spell out a given single-digit number.

#### **The Solution (Python 3)**
```python
def count_letters(num: int) -> int:
    lengths = {0:4, 1:3, 2:3, 3:5, 4:4, 5:4, 6:3, 7:5, 8:5, 9:4}
    return lengths.get(num, 0)
```

# Data Engineering Interview Prep: Medium Python (Complete 1-23)
## Topic: Two-Pointers, Hash Maps, Matrices, Intervals, and Dynamic Programming

---

## Table of Contents
1. [Amazon: Two Sum](#python-lesson-1-amazon---two-sum)
2. [FAANG: Longest Consecutive Sequence](#python-lesson-2-faang---longest-consecutive-sequence)
3. [Walmart: Spiral Matrix](#python-lesson-3-walmart---spiral-matrix)
4. [D.E. Shaw: Max Product of Three Numbers](#python-lesson-4-de-shaw---max-product-of-three-numbers)
5. [Amazon: Two Sum (Part 2 - Sorted)](#python-lesson-5-amazon---two-sum-part-2)
6. [Amazon: Two Sum (Part 3 - Data Structure)](#python-lesson-6-amazon---two-sum-part-3)
7. [Salesforce: Matrix Rotation](#python-lesson-7-salesforce---matrix-rotation)
8. [Google: Min Amplitude](#python-lesson-8-google---min-amplitude)
9. [Walmart: Average Subarray](#python-lesson-9-walmart---average-subarray)
10. [Databricks: k-Radius Average](#python-lesson-10-databricks---k-radius-average)
11. [FAANG: Idle GPU Days](#python-lesson-11-faang---idle-gpu-days)
12. [Swiggy: Hill Climbing](#python-lesson-12-swiggy---hill-climbing)
13. [Facebook: Video Ads Insertion](#python-lesson-13-facebook---video-ads-insertion)
14. [FAANG: Data Conference Attendees](#python-lesson-14-faang---data-conference-attendees)
15. [Google: Most Popular Integers](#python-lesson-15-google---most-popular-integers)
16. [Microsoft: Factorial Trailing Zeroes](#python-lesson-16-microsoft---factorial-trailing-zeroes)
17. [Blackstone: Generate Fractions](#python-lesson-17-blackstone---generate-fractions)
18. [AQR: Pearson Correlation Coefficient](#python-lesson-18-aqr---pearson-correlation-coefficient)
19. [Akuna Capital: Largest Contiguous Subarray Sum](#python-lesson-19-akuna-capital---largest-contiguous-subarray-sum)
20. [Amazon: Coin Change](#python-lesson-20-amazon---coin-change)
21. [Fintech: Looping Number](#python-lesson-21-fintech---looping-number)
22. [FAANG: Gift Card Satisfaction](#python-lesson-22-faang---gift-card-satisfaction)
23. [Adobe: Clock-wise Matrix Rotation](#python-lesson-23-adobe---clock-wise-matrix-rotation)

---

### **Python Lesson 1: Amazon - "Two Sum"**

#### **The Logic ($O(N)$ Time, $O(N)$ Space)**
Use a Hash Map to store the numbers you have seen and their indices. Check if the complement (`target - num`) exists in the map as you iterate.

#### **The Solution (Python 3)**
```python
def two_sum(nums: list[int], target: int) -> list[int]:
    seen = {}
    for i, num in enumerate(nums):
        complement = target - num
        if complement in seen:
            return [seen[complement], i]
        seen[num] = i
    return []
```

#### **Senior DE Perspective**
* **Broadcast Hash Joins:** This is the localized equivalent of a Broadcast Hash Join in PySpark, avoiding $O(N^2)$ cross-joins and network shuffles by using an $O(1)$ memory lookup.

---

### **Python Lesson 2: FAANG - "Longest Consecutive Sequence"**

#### **The Logic ($O(N)$ Time, $O(N)$ Space)**
Convert the array to a Hash Set. To avoid redundant work, only start counting a streak if the current number is the *start* of a sequence (`num - 1 not in set`).

#### **The Solution (Python 3)**
```python
def longest_consecutive(nums: list[int]) -> int:
    num_set = set(nums)
    longest_streak = 0
    for num in num_set:
        if num - 1 not in num_set:
            current_num = num
            current_streak = 1
            while current_num + 1 in num_set:
                current_num += 1
                current_streak += 1
            longest_streak = max(longest_streak, current_streak)
    return longest_streak
```

---

### **Python Lesson 3: Walmart - "Spiral Matrix"**

#### **The Logic ($O(M \times N)$ Time, $O(1)$ Space)**
Maintain four pointers representing the matrix boundaries (`top`, `bottom`, `left`, `right`). Peel off outer layers in a while loop.

#### **The Solution (Python 3)**
```python
def spiral_order(matrix: list[list[int]]) -> list[int]:
    result = []
    if not matrix: return result
    top, bottom = 0, len(matrix) - 1
    left, right = 0, len(matrix[0]) - 1
    
    while left <= right and top <= bottom:
        for col in range(left, right + 1):
            result.append(matrix[top][col])
        top += 1
        for row in range(top, bottom + 1):
            result.append(matrix[row][right])
        right -= 1
        if top <= bottom:
            for col in range(right, left - 1, -1):
                result.append(matrix[bottom][col])
            bottom -= 1
        if left <= right:
            for row in range(bottom, top - 1, -1):
                result.append(matrix[row][left])
            left += 1
    return result
```

#### **Senior DE Perspective**
* **Cache Misses:** Non-contiguous memory access in large ML matrices causes CPU cache misses. Use Zarr or HDF5 for spatial locality.

---

### **Python Lesson 4: D.E. Shaw - "Max Product of Three Numbers"**

#### **The Logic ($O(N)$ Time, $O(1)$ Space)**
Track the top 3 maximums and top 2 minimums (for negative multiples) in a single pass to avoid sorting.

#### **The Solution (Python 3)**
```python
def maximum_product(nums: list[int]) -> int:
    max1 = max2 = max3 = float('-inf')
    min1 = min2 = float('inf')
    
    for num in nums:
        if num > max1:
            max3, max2, max1 = max2, max1, num
        elif num > max2:
            max3, max2 = max2, num
        elif num > max3:
            max3 = num
            
        if num < min1:
            min2, min1 = min1, num
        elif num < min2:
            min2 = num
            
    return max(max1 * max2 * max3, min1 * min2 * max1)
```

---

### **Python Lesson 5: Amazon - "Two Sum (Part 2)"**

#### **The Logic ($O(N)$ Time, $O(1)$ Space)**
Because the array is sorted, use **Two Pointers** (left and right) closing inward based on whether the sum is too large or too small.

#### **The Solution (Python 3)**
```python
def twoSum(numbers: list[int], target: int) -> list[int]:
    left, right = 0, len(numbers) - 1
    while left < right:
        current_sum = numbers[left] + numbers[right]
        if current_sum == target:
            return [left + 1, right + 1] # 1-indexed
        elif current_sum < target:
            left += 1
        else:
            right -= 1
    return []
```

---

### **Python Lesson 6: Amazon - "Two Sum (Part 3)"**

#### **The Logic ($O(1)$ Add, $O(N)$ Find, $O(N)$ Space)**
Use a Hash Map to store frequencies. When finding, iterate keys to check for complements.

#### **The Solution (Python 3)**
```python
class TwoSum:
    def __init__(self):
        self.num_counts = {}

    def add(self, number: int) -> None:
        self.num_counts[number] = self.num_counts.get(number, 0) + 1

    def find(self, value: int) -> bool:
        for num in self.num_counts:
            complement = value - num
            if complement != num:
                if complement in self.num_counts:
                    return True
            elif self.num_counts[num] > 1:
                return True
        return False
```

---

### **Python Lesson 7: Salesforce - "Matrix Rotation"**

#### **The Logic ($O(N^2)$ Time, $O(1)$ Space)**
Rotate 90 degrees clockwise by **Transposing** (swapping `[i][j]` with `[j][i]`), then **Reversing** each row.

#### **The Solution (Python 3)**
```python
def rotate(matrix: list[list[int]]) -> None:
    n = len(matrix)
    for i in range(n):
        for j in range(i + 1, n):
            matrix[i][j], matrix[j][i] = matrix[j][i], matrix[i][j]
    for i in range(n):
        matrix[i].reverse()
```

---

### **Python Lesson 8: Google - "Min Amplitude"**

#### **The Logic ($O(N \log N)$ Time, $O(1)$ Space)**
Sort and compare the 4 scenarios of changing the 3 most extreme elements.

#### **The Solution (Python 3)**
```python
def minDifference(nums: list[int]) -> int:
    if len(nums) <= 4: return 0
    nums.sort()
    return min(
        nums[-1] - nums[3],
        nums[-2] - nums[2],
        nums[-3] - nums[1],
        nums[-4] - nums[0]
    )
```

---

### **Python Lesson 9: Walmart - "Average Subarray"**

#### **The Logic ($O(N)$ Time, $O(1)$ Space)**
Use a **Sliding Window** to subtract the exiting element and add the entering element, maintaining a running sum.

#### **The Solution (Python 3)**
```python
def findMaxAverage(nums: list[int], k: int) -> float:
    current_sum = max_sum = sum(nums[:k])
    for i in range(k, len(nums)):
        current_sum += nums[i] - nums[i - k]
        if current_sum > max_sum:
            max_sum = current_sum
    return max_sum / k
```

---

### **Python Lesson 10: Databricks - "k-Radius Average"**

#### **The Logic ($O(N)$ Time, $O(N)$ Space)**
Fixed sliding window of size `2k + 1`. 

#### **The Solution (Python 3)**
```python
def getAverages(nums: list[int], k: int) -> list[int]:
    n = len(nums)
    window_size = 2 * k + 1
    res = [-1] * n
    if window_size > n: return res
        
    window_sum = sum(nums[:window_size])
    res[k] = window_sum // window_size
    
    for i in range(window_size, n):
        window_sum += nums[i] - nums[i - window_size]
        res[i - k] = window_sum // window_size
    return res
```

---

### **Python Lesson 11: FAANG - "Idle GPU Days"**

#### **The Logic ($O(N \log N)$ Time, $O(N)$ Space)**
Sort intervals by start time. Merge if overlapping.

#### **The Solution (Python 3)**
```python
def activeDays(intervals: list[list[int]]) -> int:
    if not intervals: return 0
    intervals.sort(key=lambda x: x[0])
    
    merged = [intervals[0]]
    for current in intervals[1:]:
        prev = merged[-1]
        if current[0] <= prev[1]:
            prev[1] = max(prev[1], current[1])
        else:
            merged.append(current)
    return sum(end - start + 1 for start, end in merged)
```

---

### **Python Lesson 12: Swiggy - "Hill Climbing"**

#### **The Logic ($O(\log N)$ Time, $O(1)$ Space)**
Use **Binary Search** to find the peak in a mountain array.

#### **The Solution (Python 3)**
```python
def peakIndexInMountainArray(arr: list[int]) -> int:
    left, right = 0, len(arr) - 1
    while left < right:
        mid = (left + right) // 2
        if arr[mid] < arr[mid + 1]:
            left = mid + 1
        else:
            right = mid
    return left
```

---

### **Python Lesson 13: Facebook - "Video Ads Insertion"**

#### **The Logic ($O(N)$ Time, $O(N)$ Space)**
Iterate through, adding intervals before, mutating the new interval during overlap, and adding remaining after.

#### **The Solution (Python 3)**
```python
def insert(intervals: list[list[int]], newInterval: list[int]) -> list[list[int]]:
    res, i, n = [], 0, len(intervals)
    while i < n and intervals[i][1] < newInterval[0]:
        res.append(intervals[i])
        i += 1
    while i < n and intervals[i][0] <= newInterval[1]:
        newInterval[0] = min(newInterval[0], intervals[i][0])
        newInterval[1] = max(newInterval[1], intervals[i][1])
        i += 1
    res.append(newInterval)
    while i < n:
        res.append(intervals[i])
        i += 1
    return res
```

---

### **Python Lesson 14: FAANG - "Data Conference Attendees"**

#### **The Logic ($O(N \log N)$ Time, $O(N)$ Space)**
Meeting Rooms II. Separate and sort start/end times. Track active meetings with two pointers.

#### **The Solution (Python 3)**
```python
def minMeetingRooms(intervals: list[list[int]]) -> int:
    starts = sorted([i[0] for i in intervals])
    ends = sorted([i[1] for i in intervals])
    res, count, s, e = 0, 0, 0, 0
    
    while s < len(intervals):
        if starts[s] < ends[e]:
            count += 1
            s += 1
        else:
            count -= 1
            e += 1
        res = max(res, count)
    return res
```

---

### **Python Lesson 15: Google - "Most Popular Integers"**

#### **The Logic ($O(N \log K)$ Time, $O(N)$ Space)**
Use a Hash Map to count frequencies, then a Min-Heap of size `k`.

#### **The Solution (Python 3)**
```python
from collections import Counter
import heapq

def topKFrequent(nums: list[int], k: int) -> list[int]:
    count = Counter(nums)
    return heapq.nlargest(k, count.keys(), key=count.get)
```

---

### **Python Lesson 16: Microsoft - "Factorial Trailing Zeroes"**

#### **The Logic ($O(\log N)$ Time, $O(1)$ Space)**
Count factors of 5.

#### **The Solution (Python 3)**
```python
def trailingZeroes(n: int) -> int:
    zeros = 0
    while n > 0:
        n //= 5
        zeros += n
    return zeros
```

---

### **Python Lesson 17: Blackstone - "Generate Fractions"**

#### **The Logic ($O(N^2)$ Time, $O(N^2)$ Space)**
Nested loop; simplified if GCD is 1.

#### **The Solution (Python 3)**
```python
import math

def simplifiedFractions(n: int) -> list[str]:
    res = []
    for denominator in range(2, n + 1):
        for numerator in range(1, denominator):
            if math.gcd(numerator, denominator) == 1:
                res.append(f"{numerator}/{denominator}")
    return res
```

---

### **Python Lesson 18: AQR - "Pearson Correlation Coefficient"**

#### **The Logic ($O(N)$ Time, $O(1)$ Space)**
Implement the mathematical covariance/variance formula.

#### **The Solution (Python 3)**
```python
import math

def pearson_correlation(x: list[float], y: list[float]) -> float:
    n = len(x)
    mean_x, mean_y = sum(x) / n, sum(y) / n
    num = sum((x[i] - mean_x) * (y[i] - mean_y) for i in range(n))
    den_x = sum((x[i] - mean_x) ** 2 for i in range(n))
    den_y = sum((y[i] - mean_y) ** 2 for i in range(n))
    if den_x == 0 or den_y == 0: return 0.0
    return num / math.sqrt(den_x * den_y)
```

---

### **Python Lesson 19: Akuna Capital - "Largest Contiguous Subarray Sum"**

#### **The Logic ($O(N)$ Time, $O(1)$ Space)**
**Kadane's Algorithm.**

#### **The Solution (Python 3)**
```python
def maxSubArray(nums: list[int]) -> int:
    max_current = max_global = nums[0]
    for num in nums[1:]:
        max_current = max(num, max_current + num)
        if max_current > max_global:
            max_global = max_current
    return max_global
```

---

### **Python Lesson 20: Amazon - "Coin Change"**

#### **The Logic ($O(A \times C)$ Time, $O(A)$ Space)**
**Dynamic Programming.** Build solutions from bottom up.

#### **The Solution (Python 3)**
```python
def coinChange(coins: list[int], amount: int) -> int:
    dp = [float('inf')] * (amount + 1)
    dp[0] = 0
    for a in range(1, amount + 1):
        for c in coins:
            if a - c >= 0:
                dp[a] = min(dp[a], 1 + dp[a - c])
    return dp[amount] if dp[amount] != float('inf') else -1
```

---

### **Python Lesson 21: Fintech - "Looping Number"**

#### **The Logic ($O(\log N)$ Time, $O(1)$ Space)**
**Floyd's Tortoise and Hare.** Detect cycles.

#### **The Solution (Python 3)**
```python
def isHappy(n: int) -> bool:
    def get_next(number):
        total_sum = 0
        while number > 0:
            number, digit = divmod(number, 10)
            total_sum += digit ** 2
        return total_sum

    slow_runner = n
    fast_runner = get_next(n)
    while fast_runner != 1 and slow_runner != fast_runner:
        slow_runner = get_next(slow_runner)
        fast_runner = get_next(get_next(fast_runner))
    return fast_runner == 1
```

---

### **Python Lesson 22: FAANG - "Gift Card Satisfaction"**

#### **The Logic ($O(N \log N)$ Time, $O(1)$ Space)**
Sort and use Two-Pointers closing inward.

#### **The Solution (Python 3)**
```python
def twoSumLessThanK(nums: list[int], k: int) -> int:
    nums.sort()
    left, right = 0, len(nums) - 1
    max_sum = -1
    while left < right:
        current_sum = nums[left] + nums[right]
        if current_sum < k:
            max_sum = max(max_sum, current_sum)
            left += 1
        else:
            right -= 1
    return max_sum
```

---

### **Python Lesson 23: Adobe - "Clock-wise Matrix Rotation"**

#### **The Solution (Python 3)**
```python
def rotate_matrix(matrix: list[list[int]]) -> None:
    matrix[:] = [list(row) for row in zip(*matrix[::-1])]
```

# Data Engineering Interview Prep: Hard Python (Lesson 1)
## Topic: Backtracking, Depth-First Search (DFS), and Recursion

---

### **Python Lesson 1: Apple - "Word Search"**

#### **The Problem**
Given an `m x n` grid of characters `board` and a string `word`, return `True` if `word` exists in the grid. The word can be constructed from letters of sequentially adjacent cells, where adjacent cells are horizontally or vertically neighboring. The same letter cell may not be used more than once.

#### **The Logic ($O(M \times N \times 4^L)$ Time, $O(L)$ Space)**
This requires **Backtracking** via **Depth-First Search (DFS)**. 
1. We iterate through every single cell on the board. 
2. If the cell matches the first letter of our target word, we trigger a recursive DFS to check its 4 neighbors (up, down, left, right).
3. To ensure we don't reuse the same cell, we temporarily mutate the cell's value (e.g., to `#`) to mark it as "visited" during the current search path.
4. If a path fails, we **backtrack** by restoring the cell's original value so other paths can potentially use it.
*(Note: $L$ is the length of the word, and $M \times N$ is the size of the board).*

#### **The Solution (Python 3)**
```python
def exist(board: list[list[str]], word: str) -> bool:
    if not board or not board[0]:
        return False
        
    rows, cols = len(board), len(board[0])
    
    def dfs(r: int, c: int, i: int) -> bool:
        # Base Case 1: We found all the letters in the word
        if i == len(word):
            return True
            
        # Base Case 2: Out of bounds or letter doesn't match
        if (r < 0 or c < 0 or 
            r >= rows or c >= cols or 
            board[r][c] != word[i]):
            return False
            
        # Mark the current cell as visited by storing its value and mutating it
        temp = board[r][c]
        board[r][c] = '#'
        
        # Explore all 4 adjacent directions (Down, Up, Right, Left)
        found = (dfs(r + 1, c, i + 1) or
                 dfs(r - 1, c, i + 1) or
                 dfs(r, c + 1, i + 1) or
                 dfs(r, c - 1, i + 1))
                 
        # Backtrack: Restore the original cell value before returning
        board[r][c] = temp
        
        return found

    # Iterate through every cell on the board to find the starting letter
    for r in range(rows):
        for c in range(cols):
            if board[r][c] == word[0] and dfs(r, c, 0):
                return True
                
    return False
```

#### **Senior Data Engineer Perspective**
* **Recursion Limits:** While you won't often search for words in a grid as a Data Engineer, DFS is the exact underlying logic used to traverse dependency graphs. For example, when Apache Airflow calculates which tasks are blocked by upstream failures, it uses graph traversal. 
* **The Danger of the Call Stack:** In Python, recursion is dangerous because the maximum recursion depth is relatively low (usually 1,000 frames by default). If you use DFS to parse a deeply nested JSON payload (which happens constantly in NoSQL databases like MongoDB or nested Parquet structures), a deeply nested record will throw a `RecursionError` and crash the pipeline. A Senior DE will often rewrite deep recursive algorithms into iterative approaches using manual Stacks (lists) to push the memory burden from the limited call stack to the heap.